In [ ]:
import femm
femm.openfemm()  # Try to open FEMM hidden
print("✓ FEMM opened successfully")
femm.closefemm()

In [ ]:
print("=== Testing FEMM with COM initialization ===")
import pythoncom

pythoncom.CoInitialize()
print("✓ COM initialized")

import femm
femm.openfemm(0)  # Try to open FEMM hidden
print("✓ FEMM opened successfully with COM init")
femm.closefemm()
print("✓ FEMM closed successfully")

In [1]:
import femm
import os
import numpy as np

# --- 1. Start FEMM ---
# The FEMM process is started.
# A second argument can be used to make the window visible (0) or hidden (1)
femm.openfemm()

# --- 2. Problem Definition (Preprocessor) ---
# Create a new magnetics problem
femm.newdocument(0)

# Define the problem characteristics
# Frequency = 0 (DC), Units = millimeters, Type = planar, Precision = 1e-8, Depth = 15mm
femm.mi_probdef(0, 'millimeters', 'planar', 1e-8, 15)



In [3]:
# --- Define Materials ---
# Add Air to the materials library
femm.mi_addmaterial('Air', 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0)

# Add a material for the copper coil
# mu_x, mu_y, H_c, J_dc, sigma, d_lam, Phi_h, Phi_hx, Phi_hy, lam_type, lam_fill
femm.mi_addmaterial('Copper', 1, 1, 0, 0, 58, 0, 0, 0, 0, 0)


In [5]:
# --- Define Geometry ---
# Define the coil region and the surrounding air region
coil_inner_radius = 20
coil_outer_radius = 30
coil_height = 40

# Draw rectangles for the coil cross-section
femm.mi_drawrectangle(-coil_outer_radius, -coil_height/2, -coil_inner_radius, coil_height/2)
femm.mi_drawrectangle(coil_inner_radius, -coil_height/2, coil_outer_radius, coil_height/2)

In [6]:
# --- Define Circuits and Currents ---
# Add a circuit property for the coil with 100 Amps (total)
femm.mi_addcircprop('CoilCircuit', 100, 1) # 1 for series connected

# --- Add Block Labels ---
# These labels assign material properties to regions
# Air for the internal and external regions
femm.mi_addblocklabel(0, 0)
femm.mi_setblockprop('Air', 1, 0, '<None>', 0, 0, 0)

In [7]:
# Copper for the coil windings
# The third argument assigns the region to the circuit property
femm.mi_addblocklabel(-coil_inner_radius - 1, 0)
femm.mi_setblockprop('Copper', 1, 0, 'CoilCircuit', 0, 0, 1000) # 1000 turns in positive direction

femm.mi_addblocklabel(coil_inner_radius + 1, 0)
femm.mi_setblockprop('Copper', 1, 0, 'CoilCircuit', 0, 0, -1000) # 1000 turns in negative direction

In [9]:

# --- Define Boundary Conditions ---
# Create an Asymptotic Boundary Condition (ABC) for an open boundary problem
femm.mi_makeABC()

# Clear selected labels
femm.mi_clearselected()

In [10]:
# --- 3. Solve the Problem ---
# Save the file and run the solver
femm_file = "temp_solenoid.fem"
femm.mi_saveas(femm_file)
print("Running FEMM analysis...")
femm.mi_analyze()
print("Analysis complete.")

Running FEMM analysis...


Exception: error: Material properties have not
been defined for all block labels.
Cannot analyze the problem

In [ ]:






# --- 4. Data Extraction (Postprocessor) ---
print("Loading solution and extracting field data...")
femm.mo_loadsolution()

# Get the total number of nodes in the mesh. [1]
num_nodes = femm.mo_numnodes()
print(f"Total number of mesh nodes: {num_nodes}")

field_data = []

# Iterate through every node to get its position and field values
for i in range(1, num_nodes + 1):
    # Get the (x,y) coordinates of the nth node. [1]
    coords = femm.mo_getnode(i)
    x, y = coords[0], coords[1]

    # Get the magnetic flux density (Bx, By) at that coordinate. [1]
    bx, by = femm.mo_getb(x, y)

    # Calculate the magnitude of B
    b_mag = np.sqrt(bx**2 + by**2)

    field_data.append({'x': x, 'y': y, 'Bx': bx, 'By': by, 'B_mag': b_mag})

# --- 5. Cleanup ---
femm.closefemm()
os.remove(femm_file)
os.remove("temp_solenoid.ans") # The solution file

# --- Display a sample of the extracted data ---
print("\n--- Sample of Extracted Field Data ---")
for i in range(0, len(field_data), len(field_data)//10):
    node = field_data[i]
    print(f"Node at ({node['x']:.2f}, {node['y']:.2f}): Bx={node['Bx']:.4f}, By={node['By']:.4f}, |B|={node['B_mag']:.4f} T")



# FEMM Solver Test

This notebook tests the enhanced FEMM solver with grid-based field sampling and semantic mask generation.

In [ ]:
# Import required libraries
import sys
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Add the content directory to Python path

# Import FEMM solver
from content.femm_solver import CoaxialCoilSolver

In [ ]:
# Generate single sample for testing
with CoaxialCoilSolver() as solver:
    result = solver.analyze_coil(
        radius=50,  # 50mm
        turns=50,
        current=10,  # 10A
        grid_resolution=200,
        grid_bounds=(-100, 100, -100, 100),
        generate_mask=True
    )

    if result.success:
        print(f"Max B-field: {np.max(result.magnetic_field['B_magnitude']):.4f} T")
        print(f"Materials found: {result.material_labels}")
        print(f"Grid shape: {result.magnetic_field['B_magnitude'].shape}")
        print(f"Analysis time: {result.analysis_time:.2f} seconds")
        print(f"Grid bounds: {result.grid_bounds}")
        print(f"Grid resolution: {result.grid_resolution}")
    else:
        print(f"Analysis failed: {result.error_message}")

## Visualization of Results

In [ ]:
if result.success:
    # Create visualization
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # Plot B-field magnitude
    im1 = axes[0].imshow(result.magnetic_field['B_magnitude'], 
                        cmap='viridis', origin='lower')
    axes[0].set_title('B-field Magnitude (T)')
    axes[0].set_xlabel('X (grid points)')
    axes[0].set_ylabel('Y (grid points)')
    plt.colorbar(im1, ax=axes[0])
    
    # Plot semantic mask
    im2 = axes[1].imshow(result.semantic_mask, 
                        cmap='tab10', origin='lower')
    axes[1].set_title('Material Semantic Mask')
    axes[1].set_xlabel('X (grid points)')
    axes[1].set_ylabel('Y (grid points)')
    
    # Create custom legend for materials
    material_names = [v for k, v in result.material_labels.items()]
    material_ids = list(result.material_labels.keys())
    
    # Create patches for legend
    import matplotlib.patches as mpatches
    patches = [mpatches.Patch(color=plt.cm.tab10(i/len(material_ids)), 
                             label=name) for i, name in enumerate(material_names)]
    axes[1].legend(handles=patches, bbox_to_anchor=(1.05, 1), loc='upper left')
    
    # Plot B-field vectors
    xx, yy = result.grid_coordinates
    # Sample vectors for clarity (show every 10th vector)
    step = 10
    axes[2].quiver(xx[::step, ::step], yy[::step, ::step],
                  result.magnetic_field['Bx'][::step, ::step],
                  result.magnetic_field['By'][::step, ::step],
                  result.magnetic_field['B_magnitude'][::step, ::step],
                  cmap='plasma')
    axes[2].set_title('B-field Vectors')
    axes[2].set_xlabel('X (mm)')
    axes[2].set_ylabel('Y (mm)')
    axes[2].set_aspect('equal')
    
    plt.tight_layout()
    plt.show()
else:
    print("Cannot visualize - analysis failed")

## Test Data Export

In [ ]:
if result.success:
    # Test data export functionality
    import json
    
    # Create test output directory
    output_dir = Path("test_output")
    output_dir.mkdir(exist_ok=True)
    
    # Save field data
    np.savez_compressed(
        output_dir / "test_field.npz",
        Bx=result.magnetic_field['Bx'],
        By=result.magnetic_field['By'],
        B_magnitude=result.magnetic_field['B_magnitude']
    )
    
    # Save semantic mask
    np.save(output_dir / "test_mask.npy", result.semantic_mask)
    
    # Save metadata
    metadata = {
        "geometry_type": "coil",
        "parameters": {
            "radius": 50,
            "turns": 50,
            "current": 10
        },
        "grid_resolution": result.grid_resolution,
        "grid_bounds": result.grid_bounds,
        "material_labels": result.material_labels,
        "analysis_time": result.analysis_time,
        "max_field": float(np.max(result.magnetic_field['B_magnitude'])),
        "energy": result.energy,
        "flux_linkage": result.flux_linkage
    }
    
    with open(output_dir / "test_metadata.json", "w") as f:
        json.dump(metadata, f, indent=2)
    
    print(f"Test data saved to {output_dir}")
    print(f"Files created:")
    for file in output_dir.glob("*"):
        print(f"  - {file.name} ({file.stat().st_size} bytes)")
else:
    print("Cannot export data - analysis failed")

## Multiple Sample Test

In [ ]:
# Test generating multiple samples with different parameters
test_params = [
    {"radius": 30, "turns": 25, "current": 5},
    {"radius": 40, "turns": 40, "current": 8},
    {"radius": 60, "turns": 75, "current": 12}
]

results = []

with CoaxialCoilSolver() as solver:
    for i, params in enumerate(test_params):
        print(f"\nTesting sample {i+1} with parameters: {params}")
        
        # Adjust grid bounds based on coil size
        grid_size = params["radius"] * 2.5
        grid_bounds = (-grid_size, grid_size, -grid_size, grid_size)
        
        result = solver.analyze_coil(
            radius=params["radius"],
            turns=params["turns"],
            current=params["current"],
            grid_resolution=150,  # Slightly lower resolution for speed
            grid_bounds=grid_bounds,
            generate_mask=True
        )
        
        if result.success:
            max_field = np.max(result.magnetic_field['B_magnitude'])
            print(f"  ✓ Max B-field: {max_field:.4f} T")
            print(f"  ✓ Analysis time: {result.analysis_time:.2f}s")
            results.append((params, result))
        else:
            print(f"  ✗ Failed: {result.error_message}")

print(f"\nSuccessfully generated {len(results)} out of {len(test_params)} samples")

In [ ]:
# Compare results from multiple samples
if len(results) > 1:
    fig, axes = plt.subplots(2, len(results), figsize=(5*len(results), 10))
    if len(results) == 1:
        axes = axes.reshape(2, 1)
    
    for i, (params, result) in enumerate(results):
        # Plot B-field magnitude
        im1 = axes[0, i].imshow(result.magnetic_field['B_magnitude'], 
                            cmap='viridis', origin='lower')
        axes[0, i].set_title(f"B-field\nr={params['radius']}mm, I={params['current']}A")
        axes[0, i].set_xlabel('X (grid points)')
        axes[0, i].set_ylabel('Y (grid points)')
        plt.colorbar(im1, ax=axes[0, i])
        
        # Plot semantic mask
        im2 = axes[1, i].imshow(result.semantic_mask, 
                            cmap='tab10', origin='lower')
        axes[1, i].set_title(f"Material Mask\n{params['turns']} turns")
        axes[1, i].set_xlabel('X (grid points)')
        axes[1, i].set_ylabel('Y (grid points)')
    
    plt.tight_layout()
    plt.show()
else:
    print("Need at least 2 successful samples for comparison")